# Train pitch keypoint detector (Colab)

Fine-tunes `yolov8x-pose` on the `football-field-detection-f07vi` dataset (v12, 32 keypoints) so homography works **fully locally** in PerformanceAnalyzer.

**Setup (once):**
1. Runtime -> Change runtime type -> GPU (T4 is fine; A100 faster).
2. In the left **Secrets** (key) panel add `ROBOFLOW_API_KEY` (app.roboflow.com -> Settings -> API Keys).
3. (Optional) Mount Drive so `best.pt` survives the session.

In [ ]:
!pip -q install ultralytics roboflow
import ultralytics
print('ultralytics', ultralytics.__version__)

In [ ]:
try:
    from google.colab import userdata
    API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    API_KEY = None
if not API_KEY:
    import getpass
    API_KEY = getpass.getpass('ROBOFLOW_API_KEY: ')
print('key loaded:', bool(API_KEY))

In [ ]:
import os
from roboflow import Roboflow
rf = Roboflow(api_key=API_KEY)
proj = rf.workspace('roboflow-jvuqo').project('football-field-detection-f07vi')
ds = proj.version(12).download('yolov8', location='/content/field_detection')
DATA = '/content/field_detection/data.yaml'
print(open(DATA).read())

In [ ]:
# Tune these:
MODEL   = 'yolov8x-pose.pt'   # x = best accuracy; 'm' for ~3x faster
IMGSZ   = 960                 # more pixels on line markers = better kpts
EPOCHS  = 150
BATCH   = 8 if (IMGSZ >= 960 and MODEL.startswith('yolov8x')) else 16
print(f'training {MODEL} imgsz={IMGSZ} epochs={EPOCHS} batch={BATCH}')
!yolo task=pose mode=train model={MODEL} data={DATA} epochs={EPOCHS} \
    imgsz={IMGSZ} batch={BATCH} mosaic=0.0 plots=True project=/content/runs
import glob
weights = glob.glob('/content/runs/**/weights/best.pt', recursive=True)
print('BEST_PT', weights[0] if weights else 'MISSING')

In [ ]:
import glob, subprocess
w = glob.glob('/content/runs/**/weights/best.pt', recursive=True)[0]
!yolo task=pose mode=val model={w} data={DATA} split=valid

In [ ]:
import glob, shutil, os
src = glob.glob('/content/runs/**/weights/best.pt', recursive=True)[0]
out = '/content/best_pitch.pt'
shutil.copy(src, out)
# 1) save to Drive (if mounted)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copy(out, '/content/drive/MyDrive/best_pitch.pt')
    print('saved to MyDrive/best_pitch.pt')
except Exception as e:
    print('drive skip:', e)
# 2) or download to your machine
from google.colab import files
files.download(out)
print('done')

## Bring it home

Copy `best_pitch.pt` into the project:

```bash
cp best_pitch.pt runs/pose/runs/pose/field_v12/weights/best.pt
PYTHONPATH=. .venv/bin/python scripts/validate_pitch.py \
    --video data/raw/0bfacc_0.mp4 --max-frames 40
```

If RMSE improves, it is automatically used by `src/config.py` (`PITCH_KEYPOINT_MODEL_WEIGHTS`).